# Day 05 — Exception Handling, Logging & Retry
**Module 01: Python for AI Engineering**  
Vidya Sankalp | Applied GenAI Engineering

**Goal:** Learn how to handle errors safely, understand error messages, record useful logs, and retry an operation when a temporary problem occurs.

## Learning path
1. What is an exception?
2. `try` and `except`
3. Catching specific errors and reading the message with `as`
4. `finally` — code that always runs
5. `raise` — reporting invalid input
6. Logging with different levels
7. Handling a bad CSV row
8. A simple retry example

We'll begin with small examples and gradually connect them to real application scenarios. Custom exception classes and advanced retry patterns are outside today's scope.

---


## 1 — Imports

We'll import each module when we first need it. These modules are included with Python; you don't need to install them.

In [ ]:
import json
import logging
import time

---


## 2 — What is an Exception?

An **exception** is an error that happens while a program is running. For example, converting `"hello"` to an integer is not possible.

Without handling an exception, the program stops at that line. With `try/except`, we can respond to the problem and continue safely.

### Common exceptions

| Exception | Example | Meaning |
|---|---|---|
| `ValueError` | `int("hello")` | The value is not in the expected format |
| `TypeError` | `len(None)` | The value has the wrong type |
| `KeyError` | `customer["city"]` when `city` is missing | A dictionary key doesn't exist |
| `FileNotFoundError` | Opening a file that isn't there | The file couldn't be found |
| `json.JSONDecodeError` | `json.loads("{bad}")` | The text isn't valid JSON |

You don't need to memorize every exception. Start by reading the error name and message.

---


## 3 — `try` and `except`

Put code that might fail inside `try`. If the specified error happens, Python runs the matching `except` block.

```python
try:
    age = int("hello")
except ValueError:
    print("Please enter a whole number.")
```

**Important:** Catch the specific exception you expect, such as `ValueError`, rather than catching every possible error.

In [6]:
try:
     number = int("3.14")
     print(number)
except ValueError as error:
    print(type(error),":",error)
    print("That text is not a valid integer.")

<class 'ValueError'> : invalid literal for int() with base 10: '3.14'
That text is not a valid integer.


In [1]:
# This conversion fails, so we handle the ValueError.
try:
    number = int("3.14")
    print(number)
except ValueError:
    print("That text is not a valid integer.")

That text is not a valid integer.


In [8]:
# Catch a ValueError when converting text to an integer.
def safe_to_int(value):
    try:
        return int(value)
    except ValueError:
        print(f"Cannot convert {value!r} to an integer.")
        return None

print(safe_to_int("1001"))
print(safe_to_int("hello"))

1001
Cannot convert 'hello' to an integer.
None


In [9]:
# A dictionary lookup can raise KeyError; a missing or None name needs care.
def get_customer_name(customer, key):
    try:
        value = customer[key]
        return value.strip().title()
    except KeyError:
        print(f"Customer field {key!r} is missing.")
        return "Unknown"
    except AttributeError:
        print(f"Customer field {key!r} has no text value.")
        return "Unknown"

customer = {"name": "prudhvi akella", "email": None}
print(get_customer_name(customer, "name"))
print(get_customer_name(customer, "email"))
print(get_customer_name(customer, "city"))

Prudhvi Akella
Customer field 'email' has no text value.
Unknown
Customer field 'city' is missing.
Unknown


In [10]:
# Prefer specific exceptions over a broad `except Exception`.
# This function handles only the two errors we expect.
def get_customer_name_safe(customer, key):
    try:
        return customer[key].strip().title()
    except (KeyError, AttributeError):
        return "Unknown"

print(get_customer_name_safe({"name": "ravi kumar"}, "name"))
print(get_customer_name_safe({}, "name"))

Ravi Kumar
Unknown


In [11]:
# One except block can handle more than one expected exception type.
def parse_number(value):
    try:
        return float(value)
    except (ValueError, TypeError):
        print(f"Cannot convert {value!r} to a number.")
        return None

print(parse_number("205.21"))
print(parse_number("hello"))
print(parse_number(None))

205.21
Cannot convert 'hello' to a number.
None
Cannot convert None to a number.
None


In [ ]:
# TypeError example, handled so the notebook can continue.
try:
    print(float(None))
except TypeError as error:
    print(f"Conversion failed: {error}")

---


## 4 — Reading the error message with `as`

Use `except SomeError as error` to capture the exception object. Its message can help you understand what went wrong.

```python
try:
    age = int("hello")
except ValueError as error:
    print(f"Error details: {error}")
```

In [12]:
import json
# json.loads() raises JSONDecodeError when the text is not valid JSON.
bad_json = '{"category": "TRACK_ORDER", missing_quote: "high"}'

try:
    parsed = json.loads(bad_json)
except json.JSONDecodeError as error:
    print(f"Could not read JSON: {error}")
    parsed = {}

print("Parsed value:", parsed)

Could not read JSON: Expecting property name enclosed in double quotes: line 1 column 29 (char 28)
Parsed value: {}


---


## 5 — `finally`: code that always runs

The `finally` block runs whether the operation succeeds or an exception occurs. It is often used for cleanup or for recording that an operation has finished.

```python
try:
    print("Working...")
except ValueError:
    print("Something went wrong.")
finally:
    print("Finished.")
```

In [13]:
# A tiny example: finally runs even when the conversion fails.
try:
    number = int("hello")
except ValueError:
    print("Conversion failed.")
finally:
    print("This message always appears.")

Conversion failed.
This message always appears.


In [15]:
def parse_json_safely(raw_text):
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError as error:
        print(f"Invalid JSON: {error.msg}")
        return {}
    finally:
        print("JSON parsing attempt finished.")

print(parse_json_safely('{"status": "ok"}'))
print("-----------")
print(parse_json_safely('{bad json}'))

JSON parsing attempt finished.
{'status': 'ok'}
-----------
Invalid JSON: Expecting property name enclosed in double quotes
JSON parsing attempt finished.
{}


---


In [ ]:
# Same idea, written out step by step.
raw_response = '{bad json here}'

try:
    result = json.loads(raw_response)
except json.JSONDecodeError as error:
    print(f"Could not parse the response: {error.msg}")
    result = {}

print("Result:", result)

## 6 — `raise`: report invalid input

Use `raise` when your function detects a problem and cannot return a valid result. Python's built-in exceptions are enough for today's examples.

```python
def validate_age(age):
    if age < 0:
        raise ValueError("Age cannot be negative.")
    return age

print(validate_age(25))
```

When invalid input is passed, the caller can catch the `ValueError` and decide what to do.

In [30]:
def validate_customer_id(customer_id):
    try:
        customer_id = int(customer_id)
    except (ValueError, TypeError) as error:
        print("Invalid customer ID:", error)
        raise error

    if customer_id <= 0:
        raise ValueError("Customer ID must be greater than zero.")

    return customer_id

try:
    validate_customer_id("abc") #Exception
except ValueError as error:
    print(f"Invalid customer ID: {error}")
    raise error

print("-----------")
try:
    validate_customer_id("-5")
except ValueError as error:
    print(f"Invalid customer ID: {error}")
print("-----------")

# print(validate_customer_id("1001"))
#
# for test_id in ["abc", "-5"]:
#     try:
#         print("-------------------")
#         validate_customer_id(test_id)
#     except Exception as error:
#         print("Invalid ID:", error)
#         raise error

Invalid customer ID: invalid literal for int() with base 10: 'abc'
Invalid customer ID: invalid literal for int() with base 10: 'abc'
-----------
Invalid customer ID: Customer ID must be greater than zero.
-----------


In [33]:
def function_a():
    raise ValueError("Unable to convert the value to int in function_a")

def function_b():
    try:
        return function_a()
    except ValueError as error:
        raise error

def function_c():
    try:
        return function_b()
    except ValueError as error:
        raise error

try:
    print(function_c())
except ValueError as error:
    raise error

ValueError: Unable to convert the value to int in function_a

### Optional extension: how exceptions travel between functions

If a function raises an exception and doesn't catch it, Python passes it to the function that called it. This continues until some function handles it.

**Rule of thumb:** handle an error where you can make a useful decision. You don't need to study the full call-stack example to understand today's basics.

In [ ]:
def parse_customer_id(raw_id):
    customer_id = int(raw_id)
    if customer_id <= 0:
        raise ValueError("Customer ID must be greater than zero.")
    return customer_id

def build_customer_prompt(raw_id):
    try:
        customer_id = parse_customer_id(raw_id)
        return f"Prepare a prompt for customer {customer_id}."
    except ValueError as error:
        return f"Could not build prompt: {error}"

print(build_customer_prompt("1001"))
print(build_customer_prompt("abc"))

---


## 7 — The `logging` module

`print()` is useful while learning. In applications, **logging** lets us record messages with severity levels and filter which messages appear.

### Common log levels

| Level | When to use |
|---|---|
| `DEBUG` | Detailed information useful while developing |
| `INFO` | Normal events, such as starting a task |
| `WARNING` | Something unexpected happened, but the program can continue |
| `ERROR` | An operation failed |
| `CRITICAL` | A serious problem affecting the application |

A log level acts like a filter. For example, setting the level to `WARNING` hides `DEBUG` and `INFO` messages.

In [1]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(name)s %(module)s:%(lineno)d %(levelname)s: %(message)s"
)
log = logging.getLogger("project_name")

log.debug("Detailed information for debugging")
log.info("Program started")
log.warning("A value was missing; using a default")
log.error("An operation failed")
log.critical("A serious problem occurred")

2026-11-14 08:18:41,923 project_name 4277533338:9 DEBUG: Detailed information for debugging
2026-11-14 08:18:41,923 project_name 4277533338:10 INFO: Program started
2026-11-14 08:18:41,923 project_name 4277533338:11 WARNING: A value was missing; using a default
2026-11-14 08:18:41,923 project_name 4277533338:12 ERROR: An operation failed
2026-11-14 08:18:41,924 project_name 4277533338:13 CRITICAL: A serious problem occurred


In [8]:
import json
def safe_parse_llm_json(raw_response):
    log.info("Trying to parse the response.")

    try:
        result = json.loads(raw_response)
        log.info("Response parsed successfully.")
        return result
    except json.JSONDecodeError as error:
        log.error("Response was not valid JSON: %s", error.msg)
        return {}
    finally:
        log.info("Parsing attempt finished.")

log.debug(safe_parse_llm_json('{"category": "TRACK_ORDER"}'))
log.debug("-----------------")
log.debug(safe_parse_llm_json('{bad json}'))

2026-11-14 08:25:15,353 project_name 1447777431:3 INFO: Trying to parse the response.
2026-11-14 08:25:15,354 project_name 1447777431:7 INFO: Response parsed successfully.
2026-11-14 08:25:15,354 project_name 1447777431:13 INFO: Parsing attempt finished.
2026-11-14 08:25:15,355 project_name 1447777431:15 DEBUG: {'category': 'TRACK_ORDER'}
2026-11-14 08:25:15,355 project_name 1447777431:16 DEBUG: -----------------
2026-11-14 08:25:15,355 project_name 1447777431:3 INFO: Trying to parse the response.
2026-11-14 08:25:15,356 project_name 1447777431:10 ERROR: Response was not valid JSON: Expecting property name enclosed in double quotes
2026-11-14 08:25:15,356 project_name 1447777431:13 INFO: Parsing attempt finished.
2026-11-14 08:25:15,356 project_name 1447777431:17 DEBUG: {}


---


## 8 — Handling errors in CSV data

CSV rows often contain text, even for numbers. A row may also be missing a required column. We can catch these errors so one bad row doesn't stop the whole process.

In [ ]:
def parse_customer_csv_row(row):
    try:
        return {
            "customer_id": int(row["customer_id"]),
            "name": f'{row["first_name"]} {row["last_name"]}'.strip(),
            "email": row["email"].lower().strip(),
            "city": row["city"],
        }
    except KeyError as error:
        log.warning("Missing CSV column: %s", error)
        return {}
    except (ValueError, AttributeError) as error:
        log.warning("Invalid CSV value: %s", error)
        return {}

valid_row = {
    "customer_id": "1001", "first_name": "Ravi", "last_name": "Kumar",
    "email": "RAVI@example.com", "city": "Hyderabad"
}
missing_column = {"customer_id": "1002", "first_name": "Ananya"}
bad_id = {
    "customer_id": "not-a-number", "first_name": "Ananya",
    "last_name": "Sharma", "email": "a@example.com", "city": "Pune"
}

print(parse_customer_csv_row(valid_row))
print(parse_customer_csv_row(missing_column))
print(parse_customer_csv_row(bad_id))

---


## 9 — Retry when a temporary problem occurs

Some failures are temporary, such as a timeout or a short network interruption. A retry tries the operation again after waiting.

**Exponential backoff** means the wait gets longer after each failed attempt—for example, 0.1 seconds, then 0.2 seconds, then 0.4 seconds.

We'll simulate an API call so you can learn the retry pattern without connecting to a real service.

In [ ]:
def simulate_api_call(attempt):
    # Simulate two temporary failures, then a successful response.
    if attempt == 0:
        raise ConnectionError("Temporary connection problem")
    if attempt == 1:
        raise TimeoutError("The request took too long")
    return '{"status": "success"}'

In [ ]:
def call_with_retry(max_attempts=3):
    attempt = 0

    # Keep trying until the request succeeds or we run out of attempts.
    while attempt < max_attempts:
        try:
            log.info("Attempt %s of %s", attempt + 1, max_attempts)

            # Pass the current attempt number to the simulated API call.
            response = simulate_api_call(attempt)

            # Return immediately when the request succeeds.
            log.info("Request succeeded.")
            return response

        # Retry only for connection and timeout errors.
        except (ConnectionError, TimeoutError) as error:
            attempt += 1

            # If this was the last attempt, raise an error with the original
            # connection or timeout error attached as its cause.
            if attempt == max_attempts:
                log.error("All attempts failed: %s", error)
                raise RuntimeError("Request failed after retries.") from error

            # Wait longer after each failure: 0.1, 0.2, 0.4 seconds, etc.
            wait_seconds = 0.1 * (2 ** (attempt - 1))
            log.warning(
                "Attempt failed: %s. Retrying in %.1f seconds.",
                error,
                wait_seconds,
            )
            time.sleep(wait_seconds)


# Make the request and print the response returned by the successful attempt.
response = call_with_retry()
print("Final response:", response)

---


In [ ]:
# The retry demo above succeeds after two temporary failures.
# If every attempt failed, RuntimeError could be caught like this:
try:
    response = call_with_retry(max_attempts=1)
    print("Final response:", response)
except RuntimeError as error:
    print("Request could not be completed:", error)

## 10 — Filtering log levels

During development, `DEBUG` shows detailed messages. In a quieter run, set the level to `WARNING` to show only warnings and errors.

In [ ]:
logging.basicConfig(
    level=logging.WARNING,
    format="%(levelname)s: %(message)s",
    force=True,
)
log = logging.getLogger("day05")

log.debug("This debug message is hidden.")
log.info("This info message is hidden.")
log.warning("This warning is shown.")
log.error("This error is shown.")

# Restore the notebook's logging level for any later exercises.
logging.basicConfig(
    level=logging.DEBUG,
    format="%(levelname)s: %(message)s",
    force=True,
)
log = logging.getLogger("day05")

---


---
## 10 — Introduction to OpenTelemetry (optional)

So far, we've used **exceptions** to handle failures and **logging** to record what happened. In real applications—especially APIs and AI applications—we may also want to see how a request moves through different steps.

**OpenTelemetry (OTel)** is a standard for collecting observability data such as:

- **Traces:** the journey of one request through an application.
- **Spans:** individual steps within that journey, such as parsing a response or calling an API.
- **Attributes:** useful details attached to a span, such as the operation name or attempt number.

**How it relates to logging:**

- Logging records messages about events.
- Tracing shows the steps and time taken during a request.
- They work together; OpenTelemetry does not replace Python's `logging` module.

We'll keep this as an optional introduction because OpenTelemetry requires an additional package, unlike Python's built-in `logging` module.

### Step 1 — Install the OpenTelemetry packages

Run this installation cell only if you want to try the optional example. In a notebook, `%pip` installs packages into the environment used by the kernel.

```python
%pip install opentelemetry-api opentelemetry-sdk
```

For this beginner demo, spans will be printed to the notebook output. A real application can export telemetry to an observability backend.

In [ ]:
"""Print a generic, single-request AI agent trace as a span tree.

This is a runnable OpenTelemetry teaching demo. Model and tool calls are
simulated; replace those blocks with real calls to instrument an application.
Install dependencies with:

    python -m pip install opentelemetry-api opentelemetry-sdk

Expected hierarchy for the one request in this demo:

    application.request
    └── agent.handle_request
        └── agent.run
            ├── model.call                (understand the question)
            ├── tool.search                (look up information)
            ├── model.call                (write the answer)
            └── response.create
"""

from __future__ import annotations

from collections import defaultdict
from opentelemetry import trace
from opentelemetry.sdk.trace import SpanProcessor, TracerProvider
from opentelemetry.trace import Status, StatusCode

In [ ]:
class ConsoleTraceTree(SpanProcessor):
    """Collect ended spans and print their parent/child tree at the end.

    The usual console exporter prints spans one at a time as they finish.
    This processor keeps them until the request ends, then rebuilds the tree
    using each span's parent ID so the hierarchy is easy to read.
    """

    def __init__(self) -> None:
        self._spans = []

    def on_start(self, span, parent_context=None) -> None:
        """Handle the SDK notification that a span has just started.

        OpenTelemetry calls this as soon as a span begins. This tree printer
        does not need to save started spans: the SDK provides the completed
        span, including its timing, status, attributes, and parent ID, to
        ``on_end``. Keeping this method empty also avoids printing partial
        spans before the operation has finished.

        ``span`` is the new SDK span. ``parent_context`` is the context active
        when it was created; the completed span retains the parent relationship
        needed by ``print_trace`` to reconstruct the hierarchy.
        """
        pass

    def on_end(self, span) -> None:
        """Save one completed span for printing after the request finishes.

        The callback receives a read-only completed span. We keep it in memory
        so the later tree printer can group it under its parent using span IDs
        and display final duration/status/attributes. The demo handles one
        request; a long-running service should bound or clear this collection
        between requests to avoid unbounded memory growth.
        """
        self._spans.append(span)

    def shutdown(self) -> None:
        """Release processor resources when the tracer provider shuts down.

        This demo only stores spans in a Python list, so there is no file,
        network connection, or background worker to close. Implementations
        that own such resources should flush and close them here.
        """
        pass

    def force_flush(self, timeout_millis: int = 30000) -> bool:
        """Report that buffered telemetry is ready before printing/exiting.

        The SDK can call this to ask processors to export anything they have
        buffered. This processor does not send data elsewhere; ``on_end`` has
        already saved every completed span in ``self._spans``. Returning True
        means there is no pending asynchronous work. A real exporter should
        honor ``timeout_millis`` and return False if it cannot flush in time.
        """
        return True

    def print_trace(self) -> None:
        if not self._spans:
            print("No spans were recorded.")
            return

        # This demo runs one request, so all collected spans belong to one
        # trace. A child's parent span ID matches the parent's span ID; use
        # that relationship to rebuild the hierarchy.
        children = defaultdict(list)
        roots = []
        for span in self._spans:
            parent_id = span.parent.span_id if span.parent is not None else None
            if parent_id is None:
                roots.append(span)
            else:
                children[parent_id].append(span)

        def sort_key(span):
            return span.start_time or 0

        def render(span, prefix="", last=True):
            # This function prints one span, then calls itself for that span's
            # children. `prefix` contains the tree lines needed to keep nested
            # spans aligned. `last` says whether this span is the final sibling
            # under its parent; it chooses the correct connector below.
            branch = "└── " if last else "├── "

            # Every completed span has a status code. We convert the enum to
            # its readable name, such as OK, ERROR, or UNSET. UNSET means the
            # code did not explicitly assign a status; it does not itself mean
            # the operation failed.
            status = span.status.status_code.name

            # OpenTelemetry stores start_time and end_time as nanoseconds since
            # the Unix epoch. Subtracting gives a duration in nanoseconds;
            # divide by 1,000,000 to display milliseconds.
            elapsed_ms = (span.end_time - span.start_time) / 1_000_000

            # `attributes` is optional metadata attached to this operation.
            # Convert the mapping to a normal dict so it is easy to inspect and
            # format. The fallback handles spans with no attributes.
            attrs = dict(span.attributes or {})

            # Build the attribute text only when attributes exist. repr (`!r`)
            # makes strings visibly quoted, which helps distinguish values
            # when reading the output. Keep these values short and safe: do not
            # attach secrets, full prompts, or raw query results in real code.
            visible_attrs = ""
            if attrs:
                visible = ", ".join(f"{key}={value!r}" for key, value in attrs.items())
                visible_attrs = f" {{{visible}}}"

            # Print this span's tree connector, name, status, elapsed time, and
            # optional attributes together on one line.
            print(
                f"{prefix}{branch}{span.name} [{status}, {elapsed_ms:.1f} ms]"
                f"{visible_attrs}"
            )

            # Find spans whose parent span ID matches this span's own ID. Sort
            # them by start time so siblings appear in the order their work
            # began, rather than the reverse order in which nested spans end.
            descendants = sorted(children.get(span.context.span_id, []), key=sort_key)

            # For a non-final sibling, keep a vertical line in the next level
            # (`│   `) to show that more siblings follow. For the final sibling,
            # use spaces so the tree visually ends at this branch.
            child_prefix = prefix + ("    " if last else "│   ")
            for index, child in enumerate(descendants):
                # The final child gets `└──`; earlier children get `├──`.
                # Pass the updated prefix so deeper descendants line up below
                # this span and continue the tree correctly.
                render(child, child_prefix, index == len(descendants) - 1)

        # A root has no parent span. Print roots in start-time order as well.
        # In this teaching demo there is one request and normally one root;
        # the loop also handles multiple roots if the collector is reused.
        for index, root in enumerate(sorted(roots, key=sort_key)):
            # Treat the final root like the final sibling so its connector uses
            # `└──`. With only one root, this is always True.
            render(root, "", index == len(roots) - 1)

In [ ]:
# Configure the OpenTelemetry SDK once when the program starts. The processor
# receives every span created by this tracer and saves it for tree printing.
# A production service would usually also export spans to an OTLP collector.
console_tree = ConsoleTraceTree()
provider = TracerProvider()
provider.add_span_processor(console_tree)
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("demo.agent")

In [ ]:
def simulated_model_call(step: str) -> None:
    """Stand in for one model request made by the agent."""
    # Since this function runs inside the agent span, this model span becomes
    # its child automatically. OpenTelemetry carries the active span context.
    with tracer.start_as_current_span("model.call") as span:
        span.set_attribute("agent.step", step)
        span.set_status(Status(StatusCode.OK))

In [ ]:
def run_agent(question: str) -> dict:
    """Simulate one question handled by an AI agent."""
    # This is the request's root span. In a deployed AgentCore service, an
    # incoming server span may be the root instead; this makes the demo
    # self-contained and guarantees one trace for this one request.
    with tracer.start_as_current_span("application.request") as request_span:
        request_span.set_attribute("request.type", "question")
        request_span.set_attribute("request.length", len(question))
        request_span.set_status(Status(StatusCode.OK))

        with tracer.start_as_current_span("agent.handle_request") as handle_span:
            handle_span.set_status(Status(StatusCode.OK))

            # The agent span covers its full reasoning/tool cycle. Each nested
            # with-block below represents one operation within that cycle.
            with tracer.start_as_current_span("agent.run") as agent_span:
                agent_span.set_status(Status(StatusCode.OK))

                # The model first interprets the question and plans a response.
                simulated_model_call("understand the question")

                # The agent asks a tool to retrieve useful information.
                with tracer.start_as_current_span("tool.search") as span:
                    span.set_attribute("search.result_count", 3)
                    span.set_status(Status(StatusCode.OK))

                # The model uses the search results to write a final answer.
                simulated_model_call("write the answer")

                # The application packages the answer for its caller.
                with tracer.start_as_current_span("response.create") as span:
                    span.set_attribute("response.format", "text")
                    span.set_status(Status(StatusCode.OK))

            return {"status": "success"}

In [ ]:
# Run exactly one request. Running this function again creates a new trace
# with a different trace ID, just like handling another user request.
result = run_agent("Find helpful information about solar energy")
provider.force_flush()
# Print only after all spans have ended, so parent and child spans are
# available together and can be rendered as one hierarchy.
print("A trace is the full request. Each indented span is one operation inside it.")
print("Trace hierarchy:")
console_tree.print_trace()
print(f"\nAgent result: {result}")
provider.shutdown()

### What to notice

- `start_as_current_span(...)` creates a span for one operation.
- `set_attribute(...)` adds searchable context to the span.
- `record_exception(...)` attaches exception details to the span.
- `set_status(...)` marks the span as failed.
- The example still uses `logging`; the trace adds a view of the operation itself.

**Remember:** Start with exceptions and logging first. Treat this OpenTelemetry example as a preview of how production AI applications can be observed.

## 11 — Summary — Day 05

| Concept | Remember |
|---|---|
| Exception | An error that occurs while a program is running |
| `try/except` | Run risky code and handle an expected error |
| Specific exceptions | Catch the error types you expect, such as `ValueError` |
| `except ... as error` | Capture the error message |
| `finally` | Runs whether the operation succeeds or fails |
| `raise` | Signal that an input or operation is invalid |
| `logging` | Record events with severity levels |
| `DEBUG` / `INFO` | Development details and normal events |
| `WARNING` / `ERROR` | Unexpected situations and failures |
| CSV row handling | Catch missing columns and invalid values |
| Retry | Try again after a temporary failure |
| Exponential backoff | Increase the wait between failed attempts |

## Practice

1. Convert `"42"` to an integer inside `try/except`.
2. Handle a missing `"city"` key in a dictionary.
3. Write a function that raises `ValueError` when a price is negative.
4. Log a warning when a CSV row is missing a required field.
5. Explain why a timeout may be worth retrying, while invalid input usually isn't.

**Next:** Day 06 — Decorators and Higher-Order Functions.\n\n**Optional preview:** OpenTelemetry adds traces and spans to help observe application workflows.